# STAR testing - Hamilton STAR / STARLet

Validation notebook for the v1 STAR (`pylabrobot.hamilton.star`).

It drives a `STARDevice` - the instrument as a resource, with its deck as its child. The driver
stays reachable underneath as `star.driver`, and machine-level reads that the device does not
proxy are sent through it.

**`setup()` moves the machine.** Watch the log: it reports each phase at `DEBUG` and the machine
it found at `INFO`. It runs in three steps:

1. **discover** - read-only. Machine configuration, arm geometry, channel count, and every
   channel's firmware, width and installed hardware.
2. **initialize** - `C0 VI` on a machine that is not initialized, which homes every drive; or
   `C0 ZA` alone on one that is, to raise the channels to Z safety.
3. **capability bring-up** - the channels eject whatever is mounted on them, including grippers.

If you want to connect and look without anything moving, run `discover()` on its own; the cell
below shows how.

Set `protocol_mode` to `"simulation"` to run every cell against a simulated STAR - no hardware,
no USB, and the same code paths as the real driver.

## 1- Run identity

In [1]:
# --- Run identity ---
protocol_mode = "simulation"  # simulation OR execution
user_name = "star_user"
run_identifier = "star_v1_validation"

# --- Which instrument ---
# One of the factories in pylabrobot.hamilton.star.device. It fixes the machine's footprint and
# where its deck sits inside it, and builds the matching deck.
instrument = "STAR"  # STAR OR STARLet OR STAR_with_extension_housing

# --- Device selection (only needed with more than one Hamilton on USB) ---
device_address = None  # USB address, e.g. 3
serial_number = None  # USB serial, e.g. "1234567"

# --- Motion ---
# The X-arm move near the end of this notebook only runs when this is True.
allow_x_arm_move = False

# --- 96-head ---
# Where the 96-head ejects when it is initialized: head channel A1, in deck mm. Initializing it
# throws off whatever is mounted, so this has to be somewhere tips may be dropped, which depends
# on where the waste sits on this deck - hence no default. Setup initializes the head when this is
# set, and reports that it cannot when it is None. This machine was last sent (-263.8, 108.3,
# 200.0), read off the `C0 EI` command in an earlier run.
head96_initialize_position = (-263.8, 108.3, 200.0)

## 2- Imports

In [2]:
from pylabrobot.hamilton.star.device import STAR, STAR_with_extension_housing, STARLet
from pylabrobot.hamilton.star.driver.features.head96 import Head96
from pylabrobot.hamilton.star.driver.master import STARDriver

## 3- Logging

Uses PyLabRobot's own `setup_logger`, exactly as every other PLR run does: a single
date-stamped file per day, appended to across runs. Both the file and the notebook are at
`IO` level, so every byte sent to and received from the machine is visible and recorded.

Re-running this cell is safe: `setup_logger` replaces the file handler and `verbose`
replaces the console handler, rather than stacking a second one of each.

In [3]:
import logging

import pylabrobot
from pylabrobot.io import LOG_LEVEL_IO

log_dir = f"_logs/{protocol_mode}"

# PLR's own logger setup: one date-stamped file per day, appended to across runs. Re-running this
# cell replaces the file handler rather than stacking a second one, so lines are never duplicated.
pylabrobot.setup_logger(log_dir, level=LOG_LEVEL_IO)

# Console at IO level too: every byte sent and received appears in the notebook.
pylabrobot.verbose(True, level=LOG_LEVEL_IO)

print(f"appending to {log_dir}/pylabrobot-<YYYYMMDD>.log")
logging.getLogger("pylabrobot").info("--- %s (%s) ---", run_identifier, protocol_mode)

2026-08-19 11:39:45,056 - pylabrobot - INFO - --- star_v1_validation (simulation) ---


appending to _logs/simulation/pylabrobot-<YYYYMMDD>.log


## 4- Connect and bring the machine up

In simulation this is a `STARSimulationDriver`, which answers as a real instrument does - the
same command assembly, error decoding and response parsing run either way.

Set `head96_initialize_position` above for setup to initialize the 96-head too; without it, setup
brings everything else up and reports that it could not do the head.

To connect **without moving anything**, replace `await star.setup()` with:

```python
await star._open()
star._connected = True
await star.discover()
```

In [4]:
build = {
  "STAR": STAR,
  "STARLet": STARLet,
  "STAR_with_extension_housing": STAR_with_extension_housing,
}[instrument]

# The instrument builds its own deck and hands it to the driver, which models the machine into it.
# A simulated one answers from that model, so it is built here rather than passed in.
if protocol_mode == "execution":
  star = build(driver=STARDriver(device_address=device_address, serial_number=serial_number))
else:
  star = build(simulation=True)

# Setup builds each capability the machine turns out to have, but not over one that is already
# there - so a capability configured here keeps its configuration. In simulation the head is
# already there and answers for itself, so configure that one rather than replacing it.
if head96_initialize_position is not None:
  if star.driver.head96 is None:
    star.driver.head96 = Head96(star.driver)
  star.head96.configuration.initialize_position = head96_initialize_position

await star.setup()

print(star)
# setup logs this summary at INFO; printed here too so it is the first thing you see.
print(star.driver.format_setup_summary())

2026-08-19 11:39:45,090 - pylabrobot.hamilton.star.driver.master - DEBUG - Setting up STAR on simulation (no link) ...
2026-08-19 11:39:45,092 - pylabrobot.hamilton.star.driver.master - DEBUG - [PHASE 1] Discovery
2026-08-19 11:39:45,107 - pylabrobot.hamilton.star.driver.master - DEBUG - [PHASE 2] Instrument initialization
2026-08-19 11:39:45,108 - pylabrobot.hamilton.star.driver.master - DEBUG - machine reports not initialized - running the initialization procedure (up to 300 s)
2026-08-19 11:39:45,109 - pylabrobot.hamilton.star.driver.master - DEBUG - [PHASE 3] Capability bring-up
2026-08-19 11:39:45,110 - pylabrobot.hamilton.star.driver.master - DEBUG - channels: 0 of 8 carrying tips, instrument has just been homed - initializing
2026-08-19 11:39:45,114 - pylabrobot.hamilton.star.driver.master - DEBUG - iSWAP reports itself uninitialized - initializing
2026-08-19 11:39:45,118 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0PGth2800
2026-08-19 11:39:45,120 - 

Hamilton Microlab STAR(STARSimulationDriver, 56-track deck)
[Hamilton STAR] Connected on simulation (no link)
  Firmware: master 7.6S 25 2021_11_05 (GRU C0), pipettes 4.0S j 2022-03-16, x_arm 1.4S 2012-04-25, head96 5.0S i 2021-10-22 (H0 XE167), iswap 4.1S 2011-12-19, autoload 3.4S f 2017-01-09
  Configuration: 54 slots
  Autoload: 1D barcode scanner
  Arms: 1
    left: hamilton_legacy_star_dual_rail_arm, 354.0 mm wide, travel 95.0 to 1340.2 mm, workspace -323.2 to 1517.2 mm
      channels: 8 (1000uL) | 96-head: 96 head II | 384-head: none | iSWAP: wide gripper


In [5]:
deck = star.deck
deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

In [6]:
star.driver.deck

HamiltonSTARDeck(name='deck', location=Coordinate(121.800, 116.000, 078.500), size_x=1545, size_y=653.5, size_z=900, category=deck)

## 5- The rest of the configuration

What the setup summary does not already print, plus a check of this machine's firmware stack
against the stacks this driver has been driven with before.

In [7]:
from pylabrobot.hamilton.star.driver.confirmed_firmware_versions import suggest_entry, unconfirmed

c = star.driver.configuration
print(f"wash stations         : 1={c.wash_station_1_installed}  2={c.wash_station_2_installed}")
print(f"tip waste x           : {c.tip_waste_x_position} mm")
print(
  f"iSWAP collision-free  : {c.min_iswap_collision_free_position} to "
  f"{c.max_iswap_collision_free_position} mm"
)
print(f"pip maximal y         : {c.pip_maximal_y_position} mm")
print(f"initialized           : {await star.driver.request_initialization_status()}")

# Has each of this machine's boards been driven on the firmware it reports?
new = unconfirmed(star.driver.firmware)
print()
if not new:
  print(f"firmware: all {len(star.driver.firmware)} capabilities confirmed")
else:
  print(f"firmware: {len(new)} of {len(star.driver.firmware)} capabilities not seen before.")
  print("if this machine works, add them to confirmed_firmware_versions.py:")
  for capability, version in new.items():
    print(suggest_entry(capability, version))

wash stations         : 1=False  2=False
tip waste x           : 1340.0 mm
iSWAP collision-free  : 350.0 to 1140.0 mm
pip maximal y         : 606.5 mm
initialized           : True

firmware: all 6 capabilities confirmed


## 6- The X-arms

A STAR always has a left arm and may have a right one. `star.x_arm` is the arm on a machine that
has only one, and refuses on a machine that has two.

In [8]:
for arm in (star.left_x_arm, star.right_x_arm):
  if arm is None:
    print("right: not installed")
    continue
  a = arm.configuration
  print(f"{arm.side:5s}: {a.model}   firmware {a.firmware_version}")
  print(f"       width {a.width} mm, travel {a.x_range} mm, workspace {a.workspace_range} mm")
  print(f"       wrap {a.wrap_size} mm, reference point: {a.reference_point}")
  print(
    f"       modules: pip={a.pip_installed} iswap={a.iswap_installed} "
    f"head96={a.head96_installed} xl={a.xl_channels_installed}"
  )

try:
  print(f"\nstar.x_arm -> {star.x_arm.side}")
except ValueError as e:
  print(f"\nstar.x_arm -> {e}")

left : hamilton_legacy_star_dual_rail_arm   firmware 1.4S 2012-04-25
       width 354.0 mm, travel (95.0, 1340.2) mm, workspace (-323.2, 1517.2) mm
       wrap 595.2 mm, reference point: center
       modules: pip=True iswap=True head96=True xl=False
right: not installed

star.x_arm -> left


## 7- The pipetting channels

`configuration` holds what every channel shares; `configuration.channels` holds one entry per
channel, read off the channel itself during discovery. A machine that reports no channels - a
96-head-only STAR - has no `star.pipettes` at all.

In [9]:
if star.pipettes is None:
  print("no channels installed")
else:
  p = star.pipettes.configuration
  print("shared by every channel:")
  print(f"  y drive   {p.y_drive_mm_per_increment} mm/increment")
  print(f"  z drive   {p.z_drive_mm_per_increment} mm/increment")
  print(f"  dispense  {p.dispensing_drive_uL_per_increment} uL/increment")
  print()
  print(
    f"{'ch':>3}  {'firmware':<20} {'width':>7}  {'channel':<12} {'head':<12} {'stop disc':<10} adc"
  )
  for i, ch in enumerate(p.channels):
    print(
      f"{i:>3}  {str(ch.firmware_version):<20} {str(ch.width):>7}  {str(ch.channel_type):<12} "
      f"{str(ch.head_type):<12} {str(ch.stop_disc_type):<10} {ch.pressure_adc}"
    )

shared by every channel:
  y drive   0.046302083 mm/increment
  z drive   0.01072765 mm/increment
  dispense  0.046876 uL/increment

 ch  firmware               width  channel      head         stop disc  adc
  0  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  1  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  2  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  3  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  4  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  5  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  6  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268
  7  4.0S j 2022-03-16       8.98  ML_STAR      ML_STAR      core_ii    Renesas_X9268


## 8- Sensor read: tip presence

Each channel's sleeve sensor reports whether a tip is mounted. This reads sensors; it does not
move a channel. After a full setup every channel should be empty: the channel initialization
ejects whatever was on them.

In [10]:
presence = await star.driver.request_tip_presence()
for channel, has_tip in enumerate(presence):
  print(f"  channel {channel}: {'tip' if has_tip else '-'}")

  channel 0: -
  channel 1: -
  channel 2: -
  channel 3: -
  channel 4: -
  channel 5: -
  channel 6: -
  channel 7: -


## 9- The front cover

Two read-only commands, and what they mean is exactly what this check is for.

`C0 RW` reports three inputs, the first of them the cover input. `C0 QC` reports the cover
position. Neither says whether a cover is *fitted*: the master acts only on its non-volatile
configuration, so `main_front_cover_monitoring_installed` is what decides whether the cover is
watched at all, and `star.front_cover` exists only when it is set.

This machine reports it as not installed while the cover and its switch are physically there, so
`QC` is sent raw below rather than through the capability.

**Run this three times and record what changes**: cover shut, cover open, and cover cable
disconnected. If the cover input tracks the position it is a position input; if it holds while
the position changes it is a presence input; if neither moves, the master is not reading the
switch at all - which is what a configuration that says the monitoring is not installed predicts.

That last outcome is the one that decides whether `FrontCover` is worth keeping: a machine that
answers nothing here has no cover to drive, and the capability would only ever be an empty
`request_position` on machines configured differently from this one.


In [11]:
cover_input, second_input, reserve_input = await star.driver.request_cover_input_status()
print(f"inputs        : cover={cover_input}  second={second_input}  reserve={reserve_input}")

c = star.driver.configuration
print(
  f"monitoring    : main={c.main_front_cover_monitoring_installed}"
  f"  additional={c.additional_front_cover_monitoring_installed}"
)
print(f"covers        : left={c.left_cover_installed}  right={c.right_cover_installed}")
print(f"capability    : {star.front_cover}")

# C0 QC - request cover position. Read-only, and sent raw so it answers even on a machine whose
# configuration says the monitoring is not installed.
print(f"position (raw): {await star.driver.send_raw_command('C0QCid9989')}")
if star.front_cover is not None:
  print(f"position      : {await star.front_cover.request_position()}")

2026-08-19 11:39:45,488 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0QCid9989


inputs        : cover=True  second=False  reserve=False
monitoring    : main=False  additional=False
covers        : left=False  right=False
capability    : None
position (raw): None


## 10- Move the X-arm

**This moves the arm and everything mounted on it.** Only run it with the deck clear along the
path, and only after setup has raised the channels to Z safety.

Gated on `allow_x_arm_move`, set at the top of the notebook.

In [12]:
arm = star.x_arm
print(f"travel range: {arm.configuration.x_range} mm")

target = 500.0
if allow_x_arm_move:
  await arm.move_x(target)
  print(f"moved to {target} mm")
else:
  print(f"skipped. set allow_x_arm_move = True to move to {target} mm")

# out-of-range targets are refused before anything reaches the wire
try:
  await arm.move_x(5000.0)
except ValueError as e:
  print("guard:", e)

travel range: (95.0, 1340.2) mm
skipped. set allow_x_arm_move = True to move to 500.0 mm
guard: left X-arm x=5000.0mm is outside its drive travel range [95.0, 1340.2].


In [13]:
deck.get_resource("left_x_arm").get_location_wrt(deck)

Coordinate(x=185.9, y=0.0, z=334.7)

In [14]:
await star.x_arm.request_position(), deck.get_resource("left_x_arm").get_location_wrt(deck)

(362.9, Coordinate(x=185.9, y=0.0, z=334.7))

In [15]:
# The same gate as the section above: this moves the arm.
if allow_x_arm_move:
  await star.x_arm.move_x(500.0)
  print("moved to 500.0 mm")
else:
  print("skipped. set allow_x_arm_move = True to move")

# What the machine says, and where the model puts the arm's reference point. They should agree.
position = await star.x_arm.request_position()
arm_resource = deck.get_resource("left_x_arm")
seated = arm_resource.get_location_wrt(deck)
print(f"machine: {position} mm")
print(f"model  : {seated.x + arm_resource.get_anchor(x=star.x_arm.reference_anchor).x} mm")

2026-08-19 11:39:45,540 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: X0XPla05000lr3lw7


(500.0, Coordinate(x=323.0, y=0.0, z=334.7))

In [17]:
deck.get_resource("left_x_arm").get_size_x() / 2

177.0

## 11- Raw command escape hatch

Anything not yet wrapped in a named method can be sent directly. **Only send commands you have
confirmed are read-only** - this bypasses every guard in the driver.

In [16]:
# C0 RF - request the master's firmware version. Read-only.
print(await star.driver.send_command(module="C0", command="RF"))

# the same thing as a raw string, id included
# print(await star.driver.send_raw_command("C0RFid9999"))

# Read the autoload's stored configuration, whose first field is the scanner X-drive resolution:
# 0 => 0.1 mm/step, 1 => 0.125 mm/step on a pilot-lot unit. The driver hardcodes 0.1, which is
# only right for the units that have it.
#
# Two candidates, because the read differs by autoload generation: `QU` on the later firmware,
# and the generic parameter read `RA` on the generation this machine reports. Both are
# read-only. Whichever answers, its reply is the shape a
# request_x_resolution() would parse - so print it raw.
for attempt in ("I0QUid9990", "I0RAid9991raau"):
  try:
    print(attempt, "->", await star.driver.send_raw_command(attempt))
  except Exception as e:  # noqa: BLE001 - whatever the machine says is the answer
    print(attempt, "-> refused:", e)

2026-08-19 11:38:13,868 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: C0RF
2026-08-19 11:38:13,871 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: I0QUid9990
2026-08-19 11:38:13,872 - pylabrobot.hamilton.star.driver.simulator - IO - [simulation] write: I0RAid9991raau


None
I0QUid9990 -> None
I0RAid9991raau -> None


## 12- What is ported, and what is not

Every module hangs off the same reply router, so each remaining one is a module to add rather
than new plumbing.

| Module | Node | State |
|---|---|---|
| Master | `C0` | configuration, initialization, tip presence |
| Pipetting channels | `P1`-`PG` | firmware, width, installed hardware, initialization |
| X-drives | `X0` | firmware, absolute move |
| 96-head | `H0` | firmware, hardware, drive parameters, retract, initialization |
| iSWAP | `R0` | not ported |
| Autoload | `I0` | not ported |
| Wash stations, pumps | `W1`/`W2`, `HW`/`HU`/`HV` | not ported |

On a machine with a 96-head, setup retracts it to Z safety and probes how far it reaches. If the
head reports itself uninitialized, setup says so rather than guessing: initializing it ejects
whatever is mounted, so it needs the position to eject at - `head96.initialize(x, y, z)`.

## 13- Teardown

In [17]:
await star.stop()
print("disconnected. connected:", star.driver.connected, "| setup done:", star.driver.setup_done)

# The log is append-only and stays open for the rest of the session - nothing to close.

disconnected. connected: False | setup done: False
